# Week 5 — Customer Sales Analysis
Learn Pandas advanced manipulation: grouping, filtering, strings, dates, merge, and pivot tables.

In [ ]:
import pandas as pd
from pathlib import Path
DATA = Path('data')
sales = pd.read_csv(DATA / 'sales_data.csv')
customers = pd.read_csv(DATA / 'customer_churn.csv')
sales.head()

## 1. Explore and clean sales data

In [ ]:
sales.info()
sales.isna().sum()

In [ ]:
sales['Date'] = pd.to_datetime(sales['Date'], errors='coerce')
for col in ['Product', 'Region', 'Customer_ID']:
    sales[col] = sales[col].astype('string').str.strip()
sales['Product'] = sales['Product'].str.title()
sales['Region'] = sales['Region'].str.title()
sales['Customer_ID'] = sales['Customer_ID'].str.upper()
for col in ['Quantity', 'Price', 'Total_Sales']:
    sales[col] = pd.to_numeric(sales[col], errors='coerce')
sales = sales.dropna(subset=['Date','Product','Region','Customer_ID','Quantity','Price','Total_Sales'])
sales = sales[(sales.Quantity > 0) & (sales.Price >= 0) & (sales.Total_Sales >= 0)].drop_duplicates()
sales.head()

## 2. Datetime parts and string transformation

In [ ]:
sales['Year'] = sales['Date'].dt.year
sales['Month'] = sales['Date'].dt.month
sales['Day'] = sales['Date'].dt.day
sales['Month_Label'] = sales['Date'].dt.to_period('M').astype(str)
sales['Product_Clean'] = sales['Product'].str.replace(r'[^A-Za-z0-9 ]','',regex=True).str.strip()
sales[['Date','Year','Month','Day','Product','Product_Clean']].head()

## 3. Grouping and multiple aggregations

In [ ]:
product_summary = sales.groupby('Product_Clean').agg(
    Total_Sales=('Total_Sales','sum'), Quantity=('Quantity','sum'),
    Average_Order=('Total_Sales','mean'), Orders=('Customer_ID','count')).sort_values('Total_Sales',ascending=False)
region_summary = sales.groupby('Region').agg(Total_Sales=('Total_Sales','sum'),
    Orders=('Customer_ID','count'), Average_Sales=('Total_Sales','mean')).sort_values('Total_Sales',ascending=False)
monthly_summary = sales.groupby('Month_Label').agg(Monthly_Sales=('Total_Sales','sum'),
    Orders=('Customer_ID','count'), Quantity=('Quantity','sum'))
display(product_summary, region_summary, monthly_summary)

## 4. Filtering using AND / OR

In [ ]:
# AND: North region and sales at least median value
high_value_north = sales[(sales.Region == 'North') & (sales.Total_Sales >= sales.Total_Sales.median())]
display(high_value_north)
\n# OR: product is Laptop or Phone, OR region is West
selected = sales[(sales.Product.isin(['Laptop','Phone'])) | (sales.Region == 'West')]
display(selected.head())

## 5. Pivot table

In [ ]:
sales_pivot = pd.pivot_table(sales, index='Region', columns='Product_Clean',
    values='Total_Sales', aggfunc='sum', fill_value=0, margins=True)
display(sales_pivot)

## 6. Customer dataset and demonstration merge
**Important data-quality note:** sales Customer_ID values are `CUST###`, while churn CustomerID values are `C#####`. There is no shared exact key in these uploaded files. A sequence-based crosswalk is only a classroom merge demonstration, not a verified customer identity linkage.

In [ ]:
customers['CustomerID'] = customers['CustomerID'].astype('string').str.strip().str.upper()
sales_ids = sorted(sales.Customer_ID.unique(), key=lambda x: int(''.join(ch for ch in x if ch.isdigit()) or 0))
churn_ids = customers.CustomerID.sort_values().reset_index(drop=True)
n = min(len(sales_ids), len(churn_ids))
crosswalk = pd.DataFrame({'Customer_ID': sales_ids[:n], 'CustomerID': churn_ids.iloc[:n].values})
merged = sales.merge(crosswalk, on='Customer_ID', how='left').merge(customers, on='CustomerID', how='left')
display(merged.head())

## 7. Churn overview

In [ ]:
churn_rate = customers['Churn'].mean() * 100
churn_summary = customers.groupby('Churn').agg(Customers=('CustomerID','count'),
    Average_Tenure=('Tenure','mean'), Average_Monthly_Charges=('MonthlyCharges','mean'),
    Total_Charges=('TotalCharges','sum'))
print(f'Churn rate: {churn_rate:.2f}%')
display(churn_summary)

## 8. Save summary outputs

In [ ]:
out = Path('outputs'); out.mkdir(exist_ok=True)
product_summary.to_csv(out/'product_sales_summary.csv')
region_summary.to_csv(out/'region_sales_summary.csv')
monthly_summary.to_csv(out/'monthly_sales_summary.csv')
sales_pivot.to_csv(out/'sales_pivot_table.csv')
merged.to_csv(out/'merged_customer_sales.csv', index=False)
print('Summary files exported to outputs/')